In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import yaml
from PyLTSpice import RawRead  # type: ignore

%config InlineBackend.figure_format = 'svg'

In [ ]:
# ============================================
# Load Configuration
# ============================================
with open("config.yaml", "r", encoding="utf-8") as config_file:
    config = yaml.safe_load(config_file)

# ============================================
# Output Directory Setup
# ============================================

REPORT_DIR = Path(config["paths"]["report_directory"])
REPORT_DIR.mkdir(exist_ok=True)

print(f"Reports directory ready at: {os.path.abspath(REPORT_DIR)}")

In [ ]:
# ============================================================
# Configuration
# ============================================================
raw_path = config["simulation"]["raw_file"]
REPORT_DIR.mkdir(exist_ok=True)

SETTLING_LIMIT_NS = 400
OVERSHOOT_LIMIT = 10
RINGING_LIMIT = 3

# ============================================================
# Load RAW File
# ============================================================
raw = RawRead(raw_path)

time = np.real(raw.get_trace(config["simulation"]["trace"]["time"]).get_wave(0))
vout_p = raw.get_trace(config["simulation"]["trace"]["output_positive"]).get_wave(0)
vout_n = raw.get_trace(config["simulation"]["trace"]["output_negative"]).get_wave(0)

vdiff = np.real(vout_p - vout_n)
vdiff_abs = np.abs(vdiff)

# ============================================================
# Peak Detection (polarity independent)
# ============================================================
peak_value = np.max(vdiff_abs)
main_peak_index = np.argmax(vdiff_abs)

# ============================================================
# Final Value Estimation (tail region)
# ============================================================
tail_start = int(len(vdiff_abs) * 0.9)
final_value = np.mean(vdiff_abs[tail_start:])

# Avoid division by zero
if final_value < 1e-12:
    final_value = 1e-12

# ============================================================
# Overshoot Calculation
# ============================================================
overshoot_percent = (peak_value - final_value) / final_value * 100

# ============================================================
# Settling Time (2 percent band, robust)
# ============================================================
tau_est = 95e-9
window_time = 2 * tau_est
window_size = int(window_time / (time[1] - time[0]))

threshold = 0.95  # 95 percent of samples must be inside band
epsilon = 1e-4  # small absolute tolerance

upper_bound = final_value * 1.02 + epsilon
lower_bound = final_value * 0.98 - epsilon

settling_index = None
for i in range(main_peak_index, len(vdiff_abs) - window_size):
    window = vdiff_abs[i : i + window_size]
    inside = (window <= upper_bound) & (window >= lower_bound)

    if np.mean(inside) >= threshold:
        settling_index = i
        break

if settling_index is not None:
    settling_time = time[settling_index]
else:
    settling_time = None

# ============================================================
# Ringing Detection (relative to final value)
# ============================================================
post_peak = vdiff_abs[main_peak_index:]
zero_crossings = np.where(np.diff(np.sign(post_peak - final_value)))[0]
ring_count = len(zero_crossings)

# ============================================================
# Plot
# ============================================================
plt.figure(figsize=(10, 6))
plt.plot(time * 1e9, vdiff_abs)
plt.xlabel("Time [ns]")
plt.ylabel("Absolute Differential Output [V]")
plt.title("AFE Transient Response")
plt.grid(True)
plt.tight_layout()

plt.savefig(REPORT_DIR / "AFETransientResponse.svg", format="svg")
plt.show()

# ============================================================
# Summary
# ============================================================
print("\n========== TRANSIENT REPORT ==========")
print(f"Peak Value:              {peak_value:.6f} V")
print(f"Final Value:             {final_value:.6f} V")
print(f"Overshoot:               {overshoot_percent:.2f} percent")

if settling_time is not None:
    print(f"Settling Time (2%):      {settling_time * 1e9:.2f} ns")
else:
    print("Settling Time (2%):      Not settled")

print(f"Ringing Crossings:       {ring_count}")
print("--------------------------------------")

# ============================================================
# Pass / Fail (ANA-005)
# ============================================================
if settling_time is None:
    print("RESULT: FAIL (ANA-005) - no settling")

elif settling_time * 1e9 > SETTLING_LIMIT_NS:
    print("RESULT: FAIL (ANA-005) - settling too slow")

elif overshoot_percent > OVERSHOOT_LIMIT:
    print("RESULT: FAIL (ANA-005) - overshoot too high")

elif ring_count > RINGING_LIMIT:
    print("RESULT: FAIL (ANA-005) - excessive ringing")

else:
    print("RESULT: PASS (ANA-005)")

print("======================================\n")